In [13]:
import os
import glob
import cv2
import torch
import numpy as np
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import models
import time

In [14]:
MODEL_UNET_WEIGHTS = "best_unet_model.pth"
MODEL_DN_WEIGHTS = "best_dn_model.pth"
MODEL_WTN_WEIGHTS = "best_wtn_model.pth"
MODEL_RESNET_WEIGHTS = "best_resnet_model.pth"

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Success: Using Apple Metal (MPS) acceleration.")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("Using CUDA (NVIDIA).")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU (Warning: This will be slower).")

Using CUDA (NVIDIA).


In [15]:
# --- DIRECTORY SETUP ---
INPUT_DIR = "inference_images"   # Put your 10 images in here
OUTPUT_DIR = "inference_results"      # Plt images will save here
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [16]:
from unet import UNet
import deep_watershed_transform_network as dwt_network
import direction_network_model as dn_model

In [17]:
class_mapping = {
'Adelholzener Alpenquelle Classic': 0,
'Adelholzener Alpenquelle Naturell': 1,
'Adelholzener Classic Bio Apfelschorle': 2,
'Adelholzener Classic Naturell': 3,
'Adelholzener Gourmet Mineralwasser': 4,
'Apple Braeburn Bundle': 5,
'Apple Golden Delicious': 6,
'Apple Granny Smith': 7,
'Apple Red Boskoop': 8,
'Augustiner Lagerbraeu Hell': 9,
'Augustiner Weissbier': 10,
'Avocado': 11,
'Banana Bundle': 12,
'Banana Single': 13,
'Cafe Wunderbar Espresso': 14,
'Caona Cocoa': 15,
'Carrot': 16,
'Clementine': 17,
'Clementine Single': 18,
'Coca Cola': 19,
'Coca Cola Light': 20,
'Cocoba Cocoa': 21,
'Corn Salad': 22,
'Corny Nussvoll': 23,
'Corny Nussvoll Single': 24,
'Corny Schoko Banane': 25,
'Corny Schoko Banane Single': 26,
'Cucumber': 27,
'Douwe Egberts Professional Ground Coffee': 28,
'Dr Oetker Vitalis Knuspermuesli Klassisch': 29,
'Ethiquable Gruener Tee Ceylon': 30,
'Franken Tafelreiniger': 31,
'Gepa Bio Caffe Crema': 32,
'Gepa Bio Und Fair Fencheltee': 33,
'Gepa Bio Und Fair Kamillentee': 34,
'Gepa Bio Und Fair Kraeuterteemischung': 35,
'Gepa Bio Und Fair Pfefferminztee': 36,
'Gepa Bio Und Fair Rooibostee': 37,
'Gepa Italienischer Bio Espresso': 38,
'Grapes Green Sugraone Seedless': 39,
'Grapes Sweet Celebration Seedless': 40,
'Kilimanjaro Tea Earl Grey': 41,
'Kiwi': 42,
'Koelln Muesli Fruechte': 43,
'Koelln Muesli Schoko': 44,
'Lettuce': 45,
'Orange Single': 46,
'Oranges': 47,
'Pasta Reggia Elicoidali': 48,
'Pasta Reggia Fusilli': 49,
'Pasta Reggia Spaghetti': 50,
'Pear': 51,
'Pelikan Tintenpatrone Canon': 52,
'Rocket': 53,
'Roma Vine Tomatoes': 54,
'Salad Iceberg': 55,
'Suntory Gokuri Lemonade': 56,
'Tegernseer Hell': 57,
'Vine Tomatoes': 58,
'Zucchini': 59
}
class_names = sorted(class_mapping, key=class_mapping.get)

In [18]:
def pad_to_square(image):
    w, h = image.size
    max_wh = max(w, h)
    hp = (max_wh - w) // 2
    vp = (max_wh - h) // 2
    padding = (hp, vp, max_wh - w - hp, max_wh - h - vp)
    return T.functional.pad(image, padding, 0, 'constant')

# --- 2. LOAD ALL MODELS ONCE ---
unet_model = UNet(in_channels=3, num_classes=2).to(DEVICE)
unet_model.load_state_dict(torch.load(MODEL_UNET_WEIGHTS, map_location=DEVICE))
unet_model.eval()

dn_model = dn_model.DirectionNetwork().to(DEVICE)
dn_model.load_state_dict(torch.load(MODEL_DN_WEIGHTS, map_location=DEVICE))
dn_model.eval()

wtn_model = dwt_network.WatershedTransformNetwork().to(DEVICE)
wtn_model.load_state_dict(torch.load(MODEL_WTN_WEIGHTS, map_location=DEVICE))
wtn_model.eval()

resnet_model = models.resnet18()
resnet_model.fc = torch.nn.Linear(resnet_model.fc.in_features, len(class_names))
resnet_model.load_state_dict(torch.load(MODEL_RESNET_WEIGHTS, map_location=DEVICE))
resnet_model = resnet_model.to(DEVICE)
resnet_model.eval()

geom_transform = T.Compose([
    T.Lambda(pad_to_square),
    T.Resize((448, 448)) # Restored to training resolution
])

tensor_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- BATCH PROCESSING LOOP ---
image_paths = glob.glob(os.path.join(INPUT_DIR, "*.jpg")) + glob.glob(os.path.join(INPUT_DIR, "*.png"))
print(f"Found {len(image_paths)} images. Starting batch process...")

for img_path in image_paths:
    filename = os.path.basename(img_path)
    print(f"Processing: {filename}")

    # Read Image and Calculate Scale
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f"  Failed to read {filename}, skipping.")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Calculate scaling multipliers mapping the 1024x512 grid to the original resolution
    orig_h, orig_w = img_rgb.shape[:2]
    scale_x = orig_w / 1024.0
    scale_y = orig_h / 512.0

    img_resized = cv2.resize(img_rgb, (1024, 512))
    input_tensor = TF.to_tensor(img_resized).unsqueeze(0).to(DEVICE)

    # Inference (Detection)
    with torch.no_grad():
        output = unet_model(input_tensor)
        unet_pred_map = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)

        img_float = img_resized.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_norm = (img_float - mean) / std

        mask_float = unet_pred_map.astype(np.float32)
        mask_expanded = np.expand_dims(mask_float, axis=2)

        rgb_gated = img_norm * mask_expanded
        input_np = np.concatenate([rgb_gated, mask_expanded], axis=2)

        dn_input = torch.from_numpy(input_np).permute(2, 0, 1).float().unsqueeze(0).to(DEVICE)

        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        direction_pred = dn_model(dn_input)
        energy_logits = wtn_model(direction_pred)
        wtn_pred_map = torch.argmax(energy_logits, dim=1).squeeze().cpu().numpy().astype(np.uint8)

        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()

        end_time = time.perf_counter()
        wtn_execution_time = end_time - start_time


    # Cropping & Transforms
    binary_mask = (wtn_pred_map > 0).astype(np.uint8) * 255
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask)
    print(f"[{filename}] WTN Inference Time: {wtn_execution_time:.5f} seconds | Objects Detected: {num_labels - 1}")

    object_crops = []
    valid_boxes = []

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        if area < 200: continue

        # Scale coordinates to extract crops
        real_x = int(x * scale_x)
        real_y = int(y * scale_y)
        real_w = int(w * scale_x)
        real_h = int(h * scale_y)

        # Crop directly from the original img_rgb
        crop = img_rgb[real_y:real_y+real_h, real_x:real_x+real_w]

        crop_pil = Image.fromarray(crop)
        padded_view_pil = geom_transform(crop_pil)
        input_tensor = tensor_transform(padded_view_pil)

        object_crops.append(input_tensor)
        valid_boxes.append((real_x, real_y, real_w, real_h))

    # Inference (Classification)
    final_detections = []
    if len(object_crops) > 0:
        batch_tensor = torch.stack(object_crops).to(DEVICE)

        with torch.no_grad():
            outputs = resnet_model(batch_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            confidences, predicted_classes = torch.max(probabilities, dim=1)

        for i, (class_idx, conf) in enumerate(zip(predicted_classes, confidences)):
            label = class_names[class_idx.item()]
            final_detections.append({
                "label": label,
                "conf": conf.item(),
                "bbox": valid_boxes[i]
            })

    # Visualization & Saving
    display_img = img_rgb.copy()
    legend_items = []

    for i, det in enumerate(final_detections):
        x, y, w, h = det["bbox"]
        item_id = str(i + 1)

        # Draw the bounding box and the solid background ID tag
        cv2.rectangle(display_img, (x, y), (x + w, y + h), (0, 255, 0), 2)
        (text_w, text_h), _ = cv2.getTextSize(item_id, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(display_img, (x, y - text_h - 10), (x + text_w + 10, y), (0, 255, 0), -1)
        cv2.putText(display_img, item_id, (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

        legend_items.append(f"[{item_id}] {det['label']} ({det['conf']:.1%})")

    # Setup figure
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(display_img)
    ax.axis('off')
    ax.set_title(f"Analysis: {filename} | WTN Inference Time: {wtn_execution_time:.5f}", fontsize=16)

    # Draw the legend underneath the image in columns
    MAX_ROWS_PER_COLUMN = 15

    for idx, item_text in enumerate(legend_items):
        col = idx // MAX_ROWS_PER_COLUMN
        row = idx % MAX_ROWS_PER_COLUMN
        x_pos = 0.10 + (col * 0.30)
        y_pos = 0.20 - (row * 0.025)

        fig.text(x_pos, y_pos, item_text, fontsize=11, ha='left', va='top', family='monospace')

    # Save and close
    save_path = os.path.join(OUTPUT_DIR, f"result_{filename}")
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.3, facecolor='white')
    plt.close(fig)

print("All images processed successfully!")

Found 19 images. Starting batch process...
Processing: Adelholzoner Gourmet Mineralwasser Shelf.png
[Adelholzoner Gourmet Mineralwasser Shelf.png] WTN Inference Time: 0.01356 seconds | Objects Detected: 11
Processing: Apple Braeburn Shelf.png
[Apple Braeburn Shelf.png] WTN Inference Time: 0.01357 seconds | Objects Detected: 11
Processing: Apple Braeburn.png
[Apple Braeburn.png] WTN Inference Time: 0.01396 seconds | Objects Detected: 12
Processing: Apple Red Boskoop Shelf.png
[Apple Red Boskoop Shelf.png] WTN Inference Time: 0.01215 seconds | Objects Detected: 11
Processing: Avocado Shelf.png
[Avocado Shelf.png] WTN Inference Time: 0.01454 seconds | Objects Detected: 10
Processing: Cocoba Cocoa Shelf.png
[Cocoba Cocoa Shelf.png] WTN Inference Time: 0.01394 seconds | Objects Detected: 16
Processing: Messy Shelf 1.png
[Messy Shelf 1.png] WTN Inference Time: 0.01201 seconds | Objects Detected: 39
Processing: Messy Shelf 10.png
[Messy Shelf 10.png] WTN Inference Time: 0.01265 seconds | Obje